In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import minimize
import matplotlib.pyplot as plt

class Params:
    def __init__(self, g=1.0, eps_d=1.0, kappa_b=6.0, kappa_a=0.3):
        self.g = g
        self.eps_d = eps_d
        self.kappa_b = kappa_b
        self.kappa_a = kappa_a
        self.c = np.sqrt(2.0) * g
        self.A0 = np.sqrt(eps_d / g)
        self.B0 = -np.sqrt(2.0) * eps_d / kappa_b
        self.delta = kappa_a * kappa_b / (4.0 * g**2)


def rhs(x, p: Params):
    """True vector field F(x), x=(A,B,PA,PB). Forward time = away from
    sources, toward sinks -- NOT the -F string-method convention."""
    A, B, PA, PB = x
    c, kb, ka = p.c, p.kappa_b, p.kappa_a
    Adot = c * (PA * PB - A * B) - (ka / 2) * A - ka * PA
    Bdot = (p.g / np.sqrt(2)) * (A**2 + PA**2) - p.eps_d / np.sqrt(2) - (kb / 2) * B - kb * PB
    PAdot = c * (B * PA - A * PB) + (ka / 2) * PA
    PBdot = c * A * PA + (kb / 2) * PB
    return np.array([Adot, Bdot, PAdot, PBdot])


def jacobian(x, p: Params):
    """Analytic Jacobian dF/dx, order (A,B,PA,PB). Verified against
    03_kappa_a_zero_structure.md's J_{W+} at kappa_a=0."""
    A, B, PA, PB = x
    c, kb, ka = p.c, p.kappa_b, p.kappa_a
    J = np.array([
        [-c * B - ka / 2,   -c * A,      c * PB - ka,   c * PA],
        [c * A,             -kb / 2,     c * PA,        -kb],
        [-c * PB,            c * PA,     c * B + ka / 2, -c * A],
        [c * PA,             0.0,        c * A,          kb / 2],
    ])
    return J


def H_energy(x, p: Params):
    """Conserved Hamiltonian, kappa_a != 0 case (should be 0 at all
    fixed points, and conserved along any trajectory)."""
    A, B, PA, PB = x
    g, eps_d, kb, ka = p.g, p.eps_d, p.kappa_b, p.kappa_a
    return (-np.sqrt(2) * g * A * B * PA
            + (g * A**2 / np.sqrt(2) - eps_d / np.sqrt(2) - kb * B / 2) * PB
            - kb / 2 * PB**2 + g / np.sqrt(2) * PA**2 * PB
            - ka / 2 * PA**2 - ka / 2 * A * PA)


def fixed_points(p: Params):
    """Exact closed-form fixed points for kappa_a != 0 (Wpm', S).
    M_+' is intentionally left out here -- not needed for the W+'->S leg,
    and its quadratic-root formula is in 01_model_and_notation.md if needed."""
    A0, delta, ka, g, c, kb = p.A0, p.delta, p.kappa_a, p.g, p.c, p.kappa_b
    assert delta < A0**2, "wells have merged (delta >= A0^2); Wpm' do not exist"
    Bshift = -ka / (2 * np.sqrt(2) * g)
    Wp = np.array([np.sqrt(A0**2 - delta), Bshift, 0.0, 0.0])
    Wm = np.array([-np.sqrt(A0**2 - delta), Bshift, 0.0, 0.0])
    S = np.array([0.0, p.B0, 0.0, 0.0])

    if ka!=0:
        a = (A0**2)/(2*A0**2 - 2*delta)*(1 + (2*delta)/(A0**2)-np.sqrt(1+4*(2*delta)/(A0**2)))
        P_A_value = np.sqrt(delta/(a*(a**2-1)))
        A_value = a*P_A_value
        B_value = -2*np.sqrt(2)*g/kb*A_value**2-ka/(2*np.sqrt(2)*g)
        P_B_value = -2*np.sqrt(2)*g*A_value*P_A_value/kb
        Mp = np.array([A_value, B_value, P_A_value, P_B_value])
    else : Mp = np.array([0.0, 0.0, A0, 0.0, 0.0])
    return Wp, Wm, S, Mp

def unstable_eigenbasis(fp, p: Params, tol=1e-8):
    """Return the two real vectors spanning the unstable eigenspace at
    fixed point `fp`. Handles both the real (node) case and the complex
    (focus) case -- see 05_numerics_and_open_questions.md's warning about
    naively taking .real of a conjugate pair."""
    J = jacobian(fp, p)
    w, v = np.linalg.eig(J)
    idx = np.where(w.real > tol)[0]
    if len(idx) != 2:
        raise RuntimeError(f"expected 2 unstable eigenvalues, got {len(idx)}: {w}")
    if np.abs(w[idx[0]].imag) < tol:
        # real case: two independent real eigenvectors
        u1 = v[:, idx[0]].real
        u2 = v[:, idx[1]].real
    else:
        # complex conjugate pair (focus/spiral case): use Re and Im of
        # ONE eigenvector as the two real basis vectors (they jointly
        # span the same real invariant plane)
        vec = v[:, idx[0]]
        u1 = vec.real
        u2 = vec.imag
    u1 = u1 / np.linalg.norm(u1)
    u2 = u2 / np.linalg.norm(u2)
    return u1, u2, w[idx]

def ev(t, x):
    return R_escape - np.linalg.norm(x)
ev.terminal = True
ev.direction = -1

In [ ]:
epsilon_p = 0.1
E_J = 37 * 2 * np.pi
phi_a = 0.11
phi_b = 0.204
alpha_sq = 8
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
kappa_b = 13 / 10.4
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.5 * kappa_2
eps_d = 2 * alpha_sq * g
target_name='S'
p = Params(g=g, eps_d=eps_d, kappa_b=kappa_b, kappa_a=kappa_a)
Wp, Wm, S, Mp = fixed_points(p)
target = {'S': S, 'Wm': Wm}[target_name]
B0 = p.B0
A0p = np.sqrt(p.A0**2-p.delta)
u1, u2, w = unstable_eigenbasis(Wp, p)
print(w)

In [ ]:
kappa_b_ratio_list_bare = [1,2,2.5]#,3]
kappa_b_ratio_list = []
sol_list = []
phi_star_prev = 1.7
for kappa_b_ratio in kappa_b_ratio_list_bare:
    if kappa_b_ratio == 1: phi_star_prev = 1.7
    p = Params(g=g, eps_d=eps_d, kappa_b=kappa_b*kappa_b_ratio, kappa_a=kappa_a)
    Wp, Wm, S, Mp = fixed_points(p)
    fp, target = Wp, S
    u1, u2, w_unst = unstable_eigenbasis(fp, p)   # real basis of unstable plane at Wp
    print(w_unst)

    eps0, T, R_escape = 1e-12, 450, 10.0

    def shoot(phi):
        x0 = fp + eps0 * (np.cos(phi) * u1 + np.sin(phi) * u2)
        sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
            dense_output=True, rtol=1e-12, atol=1e-12,
            events=ev)
        return np.min(np.sum((sol.y - target[:, None])**2, axis=0))

    res = minimize(shoot, x0=[phi_star_prev], method='Nelder-Mead',
                    options={'fatol': eps0})
    phi_star = res.x[0]
    print(phi_star, res.fun)
    x0 = fp + eps0 * (np.cos(phi_star) * u1 + np.sin(phi_star) * u2)
    sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='RK45',
        dense_output=True, rtol=1e-12, atol=1e-12,
        events=ev)
    if res.fun < .1: 
        sol_list.append(sol)
        kappa_b_ratio_list.append(kappa_b_ratio)
    phi_star_prev = phi_star

In [ ]:
import mpmath
mpmath.mp.dps = 50  # 50 decimal places
eps0, T, R_escape = 1e-12, 450, 10.0
def shoot_mp(phi):
    """Higher precision shoot using mpmath for the ODE integration."""
    phi = float(phi)  # mpmath scalar → Python float for solve_ivp
    x0 = fp + eps0 * (np.cos(phi) * u1 + np.sin(phi) * u2)
    sol = solve_ivp(lambda t, x: rhs(x, p), [0, T], x0, method='DOP853',
                    dense_output=True, rtol=1e-13, atol=1e-13, events=ev)
    
    return sol
    #return np.min(np.sum((sol.y - target[:, None])**2, axis=0))

def cross_function(phi):
    sol = shoot_mp(phi)    
    Bmin = np.min(sol.y[1])
    return Bmin - B0

In [ ]:
B0

In [ ]:
phi_scan = np.linspace(0, 2*np.pi, 1000)
cross_vals = [cross_function(phi) for phi in phi_scan]
sol_list = [shoot_mp(phi) for phi in phi_scan]

In [ ]:
plt.scatter(phi_scan, cross_vals)
plt.axhline(0, color='r', ls='--')
plt.axvline(6.24, color='grey', ls='--')
#plt.xlim(np.pi-.1, np.pi+.1)
plt.show()

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(20,10))
for (i,phi_1) in enumerate(phi_scan):
    y = (sol_list[i]).y
    A, B, PA, PB = y[0], y[1], y[2], y[3]
    axes[0].plot(A, PA)
    axes[1].plot(A, B)
    axes[2].plot(PA, PB)
axes[0].scatter([Wp[0]],[Wp[2]])
axes[0].scatter([S[0]],[S[2]])
axes[0].scatter([Mp[0]],[Mp[2]])
axes[1].scatter([Wp[0]],[Wp[1]])
axes[1].scatter([S[0]],[S[1]])
axes[1].scatter([Mp[0]],[Mp[1]])
axes[2].scatter([Wp[2]],[Wp[3]])
axes[2].scatter([S[2]],[S[3]])
axes[2].scatter([Mp[2]],[Mp[3]])
plt.show()


In [ ]:
import mpmath

a = mpmath.mpf('6.28')
b = mpmath.mpf('7')
ftol = mpmath.mpf('1e-6')
xtol = mpmath.mpf('1e-14')
res = mpmath.mpf('2')

while (mpmath.fabs(res) > ftol) and (mpmath.fabs(a - b) > xtol):
    print(float(a - b))
    phi = (a + b) / 2
    res = mpmath.mpf(cross_function(float(phi)))
    if res > 0:
        b = phi
    else:
        a = phi

print(f"cross_function = ({float(cross_function(float(a))):.4e},{float(cross_function(float(b))):.4e})")
print(f"bracket = ({float(a)}, {float(b)})")

In [ ]:
sol = shoot_mp(a)
A,B,PA,PB = sol.y
plt.plot(A,B)
plt.scatter([0],[B0])
plt.show()

In [ ]:
epsilon_p = 0.1
E_J = 37 * 2 * np.pi
phi_a = 0.11
phi_b = 0.204
alpha_sq = 8
g = np.sin(epsilon_p) * E_J * phi_a**2 * phi_b
kappa_b = 13 / 10.4
kappa_2 = 4 * g**2 / kappa_b
kappa_a = 0.5 * kappa_2
eps_d = 2 * alpha_sq * g
p = Params(g=g, eps_d=eps_d, kappa_b=kappa_b, kappa_a=kappa_a)
Wp, Wm, S, Mp = fixed_points(p)
fp, target = S, Wp
B0 = p.B0
A0p = np.sqrt(p.A0**2-p.delta)
u1, u2, w = unstable_eigenbasis(S, p)
print(w)

In [ ]:
import mpmath
mpmath.mp.dps = 50  # 50 decimal places
eps0, T, R_escape = 1e-12, 450, 10.0
def shoot_mp(phi):
    """Higher precision shoot using mpmath for the ODE integration."""
    phi = float(phi)  # mpmath scalar → Python float for solve_ivp
    x0 = fp + eps0 * (np.cos(phi) * u1 + np.sin(phi) * u2)
    sol = solve_ivp(lambda t, x: -rhs(x, p), [0, T], x0, method='DOP853',
                    dense_output=True, rtol=1e-13, atol=1e-13, events=ev)
    
    return sol
    #return np.min(np.sum((sol.y - target[:, None])**2, axis=0))

def cross_function(phi):
    sol = shoot_mp(phi)    
    Bmin = np.min(sol.y[1])
    return Bmin - B0

In [ ]:
phi_scan = np.linspace(0, 2*np.pi, 1000)
cross_vals = [cross_function(phi) for phi in phi_scan]
sol_list = [shoot_mp(phi) for phi in phi_scan]

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(20,10))
for (i,phi_1) in enumerate(phi_scan):
    y = (sol_list[i]).y
    A, B, PA, PB = y[0], y[1], y[2], y[3]
    axes[0].plot(A, PA)
    axes[1].plot(A, B)
    axes[2].plot(PA, PB)
axes[0].scatter([Wp[0]],[Wp[2]])
axes[0].scatter([S[0]],[S[2]])
axes[0].scatter([Mp[0]],[Mp[2]])
axes[1].scatter([Wp[0]],[Wp[1]])
axes[1].scatter([S[0]],[S[1]])
axes[1].scatter([Mp[0]],[Mp[1]])
axes[2].scatter([Wp[2]],[Wp[3]])
axes[2].scatter([S[2]],[S[3]])
axes[2].scatter([Mp[2]],[Mp[3]])
plt.show()
